# Trình Đánh Giá Hệ Thống Sinh Giọng Nói (TTS) bằng WER

Notebook này dùng để đánh giá và so sánh chất lượng (dựa trên mức độ nhận diện từ) giữa **Qwen CosyVoice** và **Coqui-TTS**.

### Cài đặt thư viện cần thiết
Bỏ comment để chạy dòng dưới nếu bạn upload notebook lên Kaggle nhé.

In [ ]:
# !pip -q install faster-whisper jiwer tqdm soundfile

### 1. Code đánh giá tự động (Evaluation Framework)

In [ ]:
import json
from pathlib import Path
import jiwer
from tqdm.auto import tqdm
import os

# Thiết lập Faster-Whisper ASR Model (Máy chủ Kaggle chạy tốt model small/base)
try:
    from faster_whisper import WhisperModel
    device = "cuda" if os.path.exists("/dev/nvidia0") else "cpu"
    compute_type = "float16" if device == "cuda" else "int8"
    asr_model = WhisperModel("small", device=device, compute_type=compute_type)
except Exception as e:
    print("Vui lòng cài đặt faster-whisper: pip install faster-whisper")
    asr_model = None

def transcribe_audio(audio_path):
    if asr_model is None: return "(asr error)"
    segments, _ = asr_model.transcribe(str(audio_path), language="vi", beam_size=5)
    text = " ".join([s.text for s in segments]).strip()
    return text

def calculate_wer(reference_text, generated_text):
    # Làm sạch văn bản (cần nâng cấp kỹ hơn tuỳ vào tiếng Việt)
    ref = reference_text.lower().replace(",", "").replace(".", "")
    hyp = generated_text.lower().replace(",", "").replace(".", "")
    try:
        return jiwer.wer(ref, hyp)
    except:
        return 1.0


### 2. Thiết lập cấu hình và sinh Audio

In [ ]:
import shutil

# Bạn chỉnh lại đường dẫn cho đúng nếu chạy trên Kaggle nhé
TEST_MANIFEST = Path("data/training_workspace/training_dataset/metadata/test_manifest.jsonl")
OUTPUT_EVAL_DIR = Path("data/reports/evaluation")
OUTPUT_EVAL_DIR.mkdir(parents=True, exist_ok=True)

# --- INTERFACES: BẠN HÃY THAY CODE VÀO ĐÂY ---

def generate_with_qwen(text: str, output_path: str):
    # TODO: Khởi tạo Qwen CosyVoice API hoặc model fine-tune ở đây.
    # Ví dụ mock:
    Path(output_path).touch() 

def generate_with_coqui(text: str, output_path: str):
    # TODO: Khởi tạo Coqui-TTS bạn đã train.
    # Ví dụ mock:
    Path(output_path).touch() 



### 3. Vòng lặp Test

In [ ]:
def run_evaluation():
    if not TEST_MANIFEST.exists():
        print(f"File không tồn tại: {TEST_MANIFEST}")
        return

    lines = open(TEST_MANIFEST, "r", encoding="utf-8").read().splitlines()
    test_cases = [json.loads(x) for x in lines if x.strip()][:20] # Đánh giá 20 câu đầu
    
    results = []
    
    for idx, item in enumerate(tqdm(test_cases, desc="Evaluating")):
        text = item["text"]
        orig_audio = item["audio_filepath"]
        
        # Sinh file Audio bằng qwen vs coqui
        qwen_audio_path = OUTPUT_EVAL_DIR / f"test_qwen_{idx}.wav"
        coqui_audio_path = OUTPUT_EVAL_DIR / f"test_coqui_{idx}.wav"
        
        # NẾU BẠN CHƯA CÓ MODEL, Ở ĐÂY SẼ LỖI (VÌ MOCK TRẢ VỀ FILE RỖNG)
        # generate_with_qwen(text, str(qwen_audio_path))
        # generate_with_coqui(text, str(coqui_audio_path))
        
        # Nếu bạn có thật, thả comment dưới ra:
        """
        whisper_qwen = transcribe_audio(qwen_audio_path)
        whisper_coqui = transcribe_audio(coqui_audio_path)
        
        wer_qwen = calculate_wer(text, whisper_qwen)
        wer_coqui = calculate_wer(text, whisper_coqui)
        
        results.append({
            "id": item.get("id", idx),
            "original_text": text,
            "wer_qwen": wer_qwen,
            "wer_coqui": wer_coqui
        })
        """
        pass
        
    print("Hoàn tất đánh giá!")
    # Thống kê kết quả
    if results:
        avg_qwen = sum(r["wer_qwen"] for r in results) / len(results)
        avg_coqui = sum(r["wer_coqui"] for r in results) / len(results)
        print(f"Average WER Qwen CosyVoice: {avg_qwen:.4f}")
        print(f"Average WER Coqui-TTS:      {avg_coqui:.4f}")
        print("\n=> LƯU Ý: WER càng nhỏ thì giọng đọc càng chuẩn (Whisper nhận dạng đúng)!")

run_evaluation()
